# ASAP8 Detection of Change analysis

Minimal setup for cross-session Detection-of-Change analyses. This notebook deliberately stops before plotting or inferential analysis.

Canonical tables created here:
- `sessions`: one row per complete processed session
- `rois`: one row per session-specific ROI, including longitudinal identity and depth group
- `spikes`: canonical `template_v1` optical-spike table
- `events`: one row per expected image cycle, with independent change/omission marker times
- `trial_index`: lightweight map from single-trial H5 rows onto `events`

Mean-response, sequence-response, single-trial, and running loaders are imported for the analysis cells that follow.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.dataset import (
    DEPTH_GROUP_ORDER,
    build_voltage_session_table,
    build_voltage_roi_table,
)
from vip_slap2_analysis.voltage.spikes import (
    DETECTOR_VERSION,
    build_spike_table,
)
from vip_slap2_analysis.behavior.change_detection import build_change_detection_events
from vip_slap2_analysis.voltage.responses import (
    build_single_trial_index,
    load_response_package,
    get_mean_response,
    get_sequence_response,
    load_single_trial_traces,
)
from vip_slap2_analysis.behavior.encoder import compute_encoder_velocity

assert DETECTOR_VERSION == "template_v1"

## Configuration

In [2]:
BASE_PATH = Path(r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics")

TARGET_MICE = [852835, 863774]
TARGET_SESSION_LABELS = None
TARGET_SESSION_IDS = None
PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

TRACE_VARIANT = "dff_robust_f0_trial"
REGISTRATION_FILENAME = "roi_identity_registration.csv"
EXCLUDE_INVALID_ROIS = True
EXPECTED_F0_SMOOTH_SEC = 60.0

FORCE_SPIKE_RECOMPUTE = False
SPIKE_KWARGS = dict(
    height_sigma=3.0,
    template_sigma=3.5,
    prominence_sigma=0.5,
)

RUNNING_KWARGS = dict(
    wheel_radius_cm=4.69,
    encoder_units="ticks",
    ticks_per_revolution=8192,
    absolute_velocity=True,
)

## Session registry

Only sessions with the full-session trace, single-trial H5, mean NPZ, sequence NPZ, corrected Bonsai log, and imaging-epoch QC are admitted. Paths to the encoder are retained when available but are not required at this stage.

In [3]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

sessions = build_voltage_session_table(
    registry,
    subject_ids=TARGET_MICE,
    paradigms=PARADIGMS,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    session_labels=TARGET_SESSION_LABELS,
    session_ids=TARGET_SESSION_IDS,
    trace_variant=TRACE_VARIANT,
    expected_f0_smooth_sec=EXPECTED_F0_SMOOTH_SEC,
)

display(sessions[[
    "subject_id", "session_id", "session_label", "session_order",
    "dmd1_depth_um", "dmd2_depth_um", "f0_smooth_sec",
    "trace_h5", "single_trial_h5", "encoder_pkl",
]])

C:\Users\andrew.shelton\Dropbox\allen institute\Python_Code\ams\ophys\vip-slap2-analysis\src\vip_slap2_analysis\voltage\dataset.py:203: UserWarning: 852835_2026-07-24_12-13-41: F0 smoother is 10 s, expected 60 s
  warnings.warn(


,subject_id,session_id,session_label,session_order,dmd1_depth_um,dmd2_depth_um,f0_smooth_sec,trace_h5,single_trial_h5,encoder_pkl
0,852835,852835_2026-07-23_14-27-27,A0,0,40.0,170.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
1,852835,852835_2026-07-24_12-13-41,A1,1,40.0,170.0,10.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
2,852835,852835_2026-07-27_11-59-54,A2,2,40.0,170.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
3,852835,852835_2026-07-28_17-13-56,B0,3,40.0,170.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
4,852835,852835_2026-07-29_10-08-37,B1,4,40.0,170.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
5,852835,852835_2026-07-30_15-04-34,B2,5,40.0,170.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
6,852835,852835_2026-08-01_14-58-35,G1,6,40.0,170.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
7,863774,863774_2026-08-04_08-12-47,A0,0,190.0,120.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
8,863774,863774_2026-08-05_13-17-41,A1,1,190.0,120.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...
9,863774,863774_2026-08-06_09-15-41,A2,2,190.0,120.0,60.0,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...


## ROI registry

The processed trace-H5 QC mask and manual registration QC are kept separately. `cell_id` uses the longitudinal `global_cell_id` when available and otherwise remains session-specific. Depth groups match the ephys notebook: `<100 µm`, `100–150 µm`, and `>150 µm`.

In [4]:
rois = build_voltage_roi_table(
    sessions,
    registration_filename=REGISTRATION_FILENAME,
    exclude_invalid_rois=EXCLUDE_INVALID_ROIS,
)

print(f"{rois['included'].sum()} / {len(rois)} ROI observations included")
display(rois)

99 / 105 ROI observations included


,subject_id,session_id,session_label,session_order,session_type,dmd,roi,depth_um,depth_group,trace_h5_valid_roi,source_roi_label,global_cell_id,registration_valid_roi,excluded,confidence,notes,valid_roi,manually_registered,cell_id,included
0,852835,852835_2026-07-23_14-27-27,A0,0,A0,1,0,40.0,<100 µm,True,DMD1_ROI0,852835_C003,True,False,high,NaN,True,True,852835_C003,True
1,852835,852835_2026-07-23_14-27-27,A0,0,A0,1,1,40.0,<100 µm,True,DMD1_ROI1,852835_C002,True,False,high,NaN,True,True,852835_C002,True
2,852835,852835_2026-07-23_14-27-27,A0,0,A0,1,2,40.0,<100 µm,True,DMD1_ROI2,852835_C001,True,False,high,NaN,True,True,852835_C001,True
3,852835,852835_2026-07-23_14-27-27,A0,0,A0,1,3,40.0,<100 µm,True,DMD1_ROI3,852835_C000,True,False,high,NaN,True,True,852835_C000,True
4,852835,852835_2026-07-23_14-27-27,A0,0,A0,2,0,170.0,>150 µm,True,DMD2_ROI0,852835_C006,True,False,high,NaN,True,True,852835_C006,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,863774,863774_2026-08-09_11-37-01,B2,5,B2,2,0,120.0,100–150 µm,True,DMD2_ROI0,863774_C003,True,False,high,NaN,True,True,863774_C003,True
101,863774,863774_2026-08-09_11-37-01,B2,5,B2,2,1,120.0,100–150 µm,False,DMD2_ROI1,863774_C002,False,False,high,NaN,False,True,863774_C002,False
102,863774,863774_2026-08-09_11-37-01,B2,5,B2,2,2,120.0,100–150 µm,True,DMD2_ROI2,863774_C001,True,False,high,NaN,True,True,863774_C001,True
103,863774,863774_2026-08-09_11-37-01,B2,5,B2,2,3,120.0,100–150 µm,True,DMD2_ROI3,863774_C004,True,False,high,NaN,True,True,863774_C004,True


## Spike extraction

Uses the same source-level `template_v1` detector as the ephys notebook. Each session is cached beside its processed voltage trace as `spikes_template_v1.parquet` (compressed CSV fallback if parquet support is unavailable), with metadata that invalidates the cache if the source H5, ROI selection, detector version, or detector parameters change.

In [5]:
spikes = build_spike_table(
    sessions,
    rois,
    force=FORCE_SPIKE_RECOMPUTE,
    detection_kwargs=SPIKE_KWARGS,
)

print(f"{len(spikes):,} spikes")
display(
    spikes.groupby(
        ["subject_id", "session_label", "dmd", "roi", "depth_group"],
        observed=True,
    ).size().rename("n_spikes").reset_index()
)

1,039,642 spikes


,subject_id,session_label,dmd,roi,depth_group,n_spikes
0,852835,A0,1,0,<100 µm,10097
1,852835,A0,1,1,<100 µm,7605
2,852835,A0,1,2,<100 µm,13160
3,852835,A0,1,3,<100 µm,8120
4,852835,A0,2,0,>150 µm,22474
...,...,...,...,...,...,...
94,863774,B2,1,3,>150 µm,13814
95,863774,B2,2,0,100–150 µm,13915
96,863774,B2,2,2,100–150 µm,5268
97,863774,B2,2,3,100–150 µm,8906


## Detection-of-Change event table

One row corresponds to one expected image cycle. `onset_sec` is the image-cycle timestamp used for image extraction. `change_onset_sec` and `omission_onset_sec` retain the special-event marker timestamps used for the longer change/omission extraction windows. This matters because a `ChangeFlash` marker can precede the changed-to image by one display frame.

The table also records expected versus actually presented sequence position, neighboring presented-image events, minute-scale time blocks, imaging epoch, and whether the complete image/change/omission extraction window was retained.

In [6]:
events = build_change_detection_events(sessions)

summary = (
    events.groupby(["subject_id", "session_label"])
    .agg(
        n_cycles=("event_id", "size"),
        n_changes=("is_change", "sum"),
        n_omissions=("is_omission", "sum"),
        retained_images=("retained_image_window", "sum"),
        retained_changes=("retained_change_window", "sum"),
        retained_omissions=("retained_omission_window", "sum"),
    )
    .reset_index()
)

display(summary)
display(events.head())

,subject_id,session_label,n_cycles,n_changes,n_omissions,retained_images,retained_changes,retained_omissions
0,852835,A0,2369,284,19,2363,284,19
1,852835,A1,2365,286,17,2363,286,17
2,852835,A2,2366,282,29,2317,277,29
3,852835,B0,2367,281,30,2364,281,30
4,852835,B1,2365,290,17,2362,290,17
5,852835,B2,2366,283,24,2323,279,23
6,852835,G1,1403,167,10,1271,156,10
7,863774,A0,2364,272,27,2362,272,27
8,863774,A1,2365,284,24,2264,272,23
9,863774,A2,2368,282,24,2332,277,23


,subject_id,session_id,session_label,session_order,event_id,event_uid,source_row,onset_sec,image_name,image_label,...,previous_presented_event_id,next_presented_event_id,previous_presented_image,next_presented_image,session_time_min,time_block,imaging_epoch,retained_image_window,retained_change_window,retained_omission_window
0,852835,852835_2026-07-23_14-27-27,A0,0,0,852835_2026-07-23_14-27-27:0,1,0.166217,stimuli\images_A\imk01057.tiff,imk01057,...,NaN,1.0,NaN,stimuli\images_A\imk01057.tiff,0.0,0,1.0,False,False,False
1,852835,852835_2026-07-23_14-27-27,A0,0,1,852835_2026-07-23_14-27-27:1,3,0.166217,stimuli\images_A\imk01057.tiff,imk01057,...,0.0,2.0,stimuli\images_A\imk01057.tiff,stimuli\images_A\imk01057.tiff,0.0,0,1.0,False,False,False
2,852835,852835_2026-07-23_14-27-27,A0,0,2,852835_2026-07-23_14-27-27:2,5,0.166217,stimuli\images_A\imk01057.tiff,imk01057,...,1.0,3.0,stimuli\images_A\imk01057.tiff,stimuli\images_A\imk01057.tiff,0.0,0,1.0,False,False,False
3,852835,852835_2026-07-23_14-27-27,A0,0,3,852835_2026-07-23_14-27-27:3,7,0.166217,stimuli\images_A\imk01057.tiff,imk01057,...,2.0,4.0,stimuli\images_A\imk01057.tiff,stimuli\images_A\imk01057.tiff,0.0,0,1.0,False,False,False
4,852835,852835_2026-07-23_14-27-27,A0,0,4,852835_2026-07-23_14-27-27:4,9,0.166217,stimuli\images_A\imk01057.tiff,imk01057,...,3.0,5.0,stimuli\images_A\imk01057.tiff,stimuli\images_A\imk01057.tiff,0.0,0,1.0,False,False,False


## Single-trial index

This reads only H5 metadata/onset vectors, not the trial traces themselves. Every stored image/change/omission trial is mapped one-to-one onto the canonical event table. A mismatch raises immediately rather than being repaired downstream with repeated nearest-onset joins.

In [7]:
trial_index = build_single_trial_index(sessions, events)

print(f"{len(trial_index):,} indexed single-trial response rows")
display(
    trial_index.groupby(
        ["subject_id", "session_label", "dmd", "event_type"]
    ).size().rename("n_trials").reset_index()
)
display(trial_index.head())

65,822 indexed single-trial response rows


,subject_id,session_label,dmd,event_type,n_trials
0,852835,A0,1,change,284
1,852835,A0,1,image,2363
2,852835,A0,1,omission,19
3,852835,A0,2,change,284
4,852835,A0,2,image,2363
...,...,...,...,...,...
73,863774,B2,1,image,2306
74,863774,B2,1,omission,17
75,863774,B2,2,change,277
76,863774,B2,2,image,2306


,session_id,dmd,event_type,image_name,trial_index,onset_sec,dataset_path,event_id,match_error_sec,matched,subject_id,session_label,session_order,event_uid
0,852835_2026-07-23_14-27-27,1,change,,0,12.344309,/DMD1/change/traces,21,0.0,True,852835,A0,0,852835_2026-07-23_14-27-27:21
1,852835_2026-07-23_14-27-27,1,change,,1,17.677052,/DMD1/change/traces,28,0.0,True,852835,A0,0,852835_2026-07-23_14-27-27:28
2,852835_2026-07-23_14-27-27,1,change,,2,26.777112,/DMD1/change/traces,40,0.0,True,852835,A0,0,852835_2026-07-23_14-27-27:40
3,852835_2026-07-23_14-27-27,1,change,,3,37.477022,/DMD1/change/traces,54,0.0,True,852835,A0,0,852835_2026-07-23_14-27-27:54
4,852835_2026-07-23_14-27-27,1,change,,4,42.810580,/DMD1/change/traces,61,0.0,True,852835,A0,0,852835_2026-07-23_14-27-27:61


## Ready for response analyses

The base is intentionally complete at this point. Later cells can load only what they need:

```python
session = sessions.iloc[0]
mean_pkg = load_response_package(session.mean_npz)
sequence_pkg = load_response_package(session.sequence_npz)

# Mean response for one source ROI/image
# t, y = get_mean_response(mean_pkg, dmd=1, source_roi=0,
#                          event_type="image", image_name=image_name)

# Sequence-position responses for one source ROI/image
# sequence = get_sequence_response(sequence_pkg, dmd=1, source_roi=0,
#                                  image_name=image_name, phase="repeated")

# Selected single trials only
# single = load_single_trial_traces(session.single_trial_h5, dmd=1,
#                                   event_type="omission", source_rois=[0])

# HARP-aligned running
# running = compute_encoder_velocity(session.encoder_pkl, **RUNNING_KWARGS)
```

The next analysis section should begin with image-entrained response motifs, using mean dF/F as the primary response-shape representation and the spike table as a complementary fast-output readout.